# QLoRA Fine-Tuning for NL→SQL (Unsloth, Colab T4)

Fine-tunes **Qwen2.5-7B-Instruct** with 4-bit QLoRA via [Unsloth](https://github.com/unslothai/unsloth) on the retail NL-to-SQL dataset.

**Runtime:** T4 GPU (Colab free tier — ~45 min for 3 epochs on 64 examples)

**Steps:**
1. Install Unsloth + dependencies
2. Clone repo and build dataset
3. QLoRA fine-tune
4. Evaluate base vs fine-tuned (execution accuracy)
5. Push adapters to Hugging Face Hub

In [ ]:
# ── 1. Install ────────────────────────────────────────────────────────────── #
# Unsloth provides 2-4× faster QLoRA training with the same memory footprint.
!pip install unsloth[colab-new] -q
!pip install datasets peft transformers accelerate tqdm sqlparse -q

In [ ]:
# ── 2. Clone repo and build dataset ──────────────────────────────────────── #
import os
!git clone https://github.com/Mojtaba-Alehosseini/lora-finetune-sql.git
os.chdir('lora-finetune-sql')
!python data/build_dataset.py

In [ ]:
# ── Hyperparameters ────────────────────────────────────────────────────────── #
MODEL_NAME   = 'Qwen/Qwen2.5-7B-Instruct'
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
LR           = 2e-4
EPOCHS       = 3
BATCH_SIZE   = 4
MAX_LENGTH   = 512
SEED         = 42
ADAPTER_DIR  = 'adapters/'
HF_REPO      = 'Mojtaba-Alehosseini/lora-finetune-sql-qwen25-7b'  # adjust if needed

In [ ]:
# ── 3. Load model with Unsloth (4-bit QLoRA) ─────────────────────────────── #
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_LENGTH,
    dtype=None,         # auto-detect: bf16 on Ampere+, fp16 on T4
    load_in_4bit=True,  # QLoRA: 4-bit NF4 + double quantization
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)
model.print_trainable_parameters()

In [ ]:
# ── Build training dataset ─────────────────────────────────────────────────── #
import json
from torch.utils.data import Dataset

def make_prompt(question, schema):
    return (
        'You are an expert SQL generator. '
        'Given a database schema and a natural language question, '
        'write a single valid SQL query that answers the question. '
        'Return only the SQL query, nothing else.\n\n'
        f'Schema: {schema}\n\n'
        f'Question: {question}\n\n'
        'SQL:'
    )

class NLSQLDataset(Dataset):
    def __init__(self, path, tokenizer, max_length):
        self.samples = [json.loads(l) for l in open(path).read().splitlines() if l.strip()]
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        prompt = make_prompt(item['question'], item['schema'])
        full = prompt + ' ' + item['sql']
        enc = self.tokenizer(full, max_length=self.max_length, truncation=True,
                             padding='max_length', return_tensors='pt')
        ids = enc['input_ids'].squeeze(0)
        mask = enc['attention_mask'].squeeze(0)
        p_len = self.tokenizer(prompt, max_length=self.max_length,
                               truncation=True, return_tensors='pt')['input_ids'].shape[1]
        labels = ids.clone()
        labels[:p_len] = -100
        labels[mask == 0] = -100
        return {'input_ids': ids, 'attention_mask': mask, 'labels': labels}

train_ds = NLSQLDataset('data/train.jsonl', tokenizer, MAX_LENGTH)
print(f'Training examples: {len(train_ds)}')

In [ ]:
# ── 4. Train ──────────────────────────────────────────────────────────────── #
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir=ADAPTER_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LR,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy='epoch',
    seed=SEED,
    report_to='none',
    dataloader_num_workers=2,
)

trainer = Trainer(model=model, args=training_args, train_dataset=train_ds)
trainer_stats = trainer.train()
print(f'Training done. Runtime: {trainer_stats.metrics["train_runtime"]:.1f}s')

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'Adapters saved to {ADAPTER_DIR}')

In [ ]:
# ── 5. Evaluate: base vs fine-tuned ──────────────────────────────────────── #
# Re-run eval_compare.py which loads base + adapters and reports execution accuracy.
!python src/eval_compare.py --adapter-dir adapters/

In [ ]:
# ── 6. Push adapters to Hugging Face Hub ─────────────────────────────────── #
# Set HF_TOKEN in Colab secrets (key icon → add secret HF_TOKEN)
import os
from huggingface_hub import login

token = os.environ.get('HF_TOKEN') or input('HF token: ')
login(token=token)

from peft import PeftModel
model_to_push = PeftModel.from_pretrained(model, ADAPTER_DIR)
model_to_push.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)
print(f'Pushed to: https://huggingface.co/{HF_REPO}')

In [ ]:
# ── 7. Optional: GGUF export for Ollama ──────────────────────────────────── #
# Requires llama.cpp compiled with CUDA. Merge first, then convert.
!git clone https://github.com/ggerganov/llama.cpp.git --depth 1
!pip install -r llama.cpp/requirements.txt -q

!python src/export_gguf.py \
    --adapter-dir adapters/ \
    --output-dir gguf/ \
    --llamacpp-convert llama.cpp/convert_hf_to_gguf.py

print('To use locally: ollama create nl-to-sql -f gguf/Modelfile && ollama run nl-to-sql')